In [24]:
# ===== Cell 2: helpers (NSE only) =====

def _to_date(d: pd.Timestamp | date) -> date:
    if isinstance(d, pd.Timestamp):
        return d.date()
    return d

def fetch_daily_nse(symbol: str, start_dt: pd.Timestamp, end_dt: pd.Timestamp, retries: int = 3, pause: float = 1.5) -> pd.DataFrame:
    """
    Fetch daily OHLC from NSE for [start_dt, end_dt] inclusive using nsepy.
    Returns DataFrame indexed by python date with a 'Close' column.
    """
    s = _to_date(start_dt)
    e = _to_date(end_dt)
    print(f"[NSE] get_history(symbol={symbol}, {s}..{e})")
    last_err = None
    for k in range(retries):
        try:
            df = get_history(symbol=symbol, start=s, end=e)
            if df is None or df.empty:
                print("  -> NSE returned 0 rows")
                return pd.DataFrame(columns=["Close"])
            # Make a clean date index
            out = df.reset_index().rename(columns={"Date":"_dt"})
            out["_dt"] = pd.to_datetime(out["_dt"]).dt.date
            out = out.set_index("_dt").sort_index()
            out = out[["Close"]].copy()
            print(f"  -> NSE rows: {len(out)} | first={out.index.min()} | last={out.index.max()}")
            return out
        except Exception as e1:
            last_err = e1
            print(f"  -> attempt {k+1}/{retries} failed: {e1}")
            time.sleep(pause * (k+1))
    # all failed
    raise SystemExit(f"NSE fetch failed for {symbol} [{s}..{e}] : {last_err}")

def pick_T_and_next3(daily: pd.DataFrame, result_ts: pd.Timestamp) -> dict:
    """
    Rule:
      - If result date exists in series => T is that day.
      - Otherwise T = closest previous trading day.
      - T+1..T+3 are simply the next 3 *rows* after T.
    """
    if daily is None or daily.empty or pd.isna(result_ts):
        return {
            "T_date": pd.NaT, "T": np.nan,
            "T1_date": pd.NaT, "T+1": np.nan,
            "T2_date": pd.NaT, "T+2": np.nan,
            "T3_date": pd.NaT, "T+3": np.nan,
        }

    idx = list(daily.index)  # list of python date
    rdate = result_ts.date()

    # exact?
    if rdate in idx:
        t_pos = idx.index(rdate)
        print(f"    -> exact match at {rdate} (pos {t_pos})")
    else:
        # closest previous
        earlier = [d for d in idx if d <= rdate]
        if not earlier:
            print(f"    -> no earlier trading day for {rdate}")
            return {
                "T_date": pd.NaT, "T": np.nan,
                "T1_date": pd.NaT, "T+1": np.nan,
                "T2_date": pd.NaT, "T+2": np.nan,
                "T3_date": pd.NaT, "T+3": np.nan,
            }
        t_date = max(earlier)
        t_pos = idx.index(t_date)
        print(f"    -> {rdate} not in series; using closest previous {t_date} (pos {t_pos})")

    # T
    T_date = idx[t_pos]
    T_val  = float(daily.loc[T_date, "Close"])
    print(f"       T: {T_date} = {T_val:,.2f}")

    # next 3 rows after T
    out = {"T_date": T_date, "T": T_val}
    for k in (1,2,3):
        pos = t_pos + k
        if pos < len(idx):
            d = idx[pos]
            v = float(daily.loc[d, "Close"])
            out[f"T{k}_date"] = d
            out[f"T+{k}"] = v
            print(f"       T+{k}: {d} = {v:,.2f}")
        else:
            out[f"T{k}_date"] = pd.NaT
            out[f"T+{k}"] = np.nan
            print(f"       T+{k}: (no next row) -> NaN")
    return out


In [25]:
# Cell 3 — Daily prices without AV/Yahoo: NSE archives first, then BSE
import pandas as pd
import requests, io, zipfile, time
from datetime import datetime, date, timedelta
from typing import Optional, Tuple
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ---- Config (TCS only for now) ----
NSE_SYMBOL = "TCS"
BSE_CODE_BY_NSE = {"TCS": "532540"}  # add more later
SCAN_BACK_DAYS = 60     # how far back we’ll look for T if exact date unavailable
SCAN_FWD_DAYS  = 30     # how far ahead we’ll look for each of T+1, T+2, T+3

# ---- Sessions with retries ----
def _mk_session(ref: str) -> requests.Session:
    s = requests.Session()
    s.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Referer": ref,
    })
    retry = Retry(
        total=5, connect=5, read=5,
        backoff_factor=0.6,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )
    adapter = HTTPAdapter(max_retries=retry)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    return s

_s_nse = _mk_session("https://www.nseindia.com/")
_s_bse = _mk_session("https://www.bseindia.com/")

# ---- NSE helpers ----
def _nse_bhav_url(d: date) -> str:
    # https://archives.nseindia.com/content/historical/EQUITIES/2025/JUL/cm10JUL2025bhav.csv.zip
    mon = d.strftime("%b").upper()
    tag = d.strftime("%d%b%Y").upper()
    return f"https://archives.nseindia.com/content/historical/EQUITIES/{d.year}/{mon}/cm{tag}bhav.csv.zip"

def _load_nse_bhavcopy(d: date) -> Optional[pd.DataFrame]:
    url = _nse_bhav_url(d)
    r = _s_nse.get(url, timeout=25)
    if r.status_code == 404:
        return None
    r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
        names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        if not names:
            return None
        with zf.open(names[0]) as f:
            df = pd.read_csv(f)
    df.columns = [c.strip().upper().replace(" ", "_") for c in df.columns]
    return df

def _extract_close_nse(df: pd.DataFrame, nse_symbol: str) -> Optional[float]:
    if df is None: 
        return None
    if "SYMBOL" not in df.columns or "CLOSE" not in df.columns:
        return None
    sub = df[(df["SYMBOL"].astype(str).str.upper() == nse_symbol.upper()) & (df.get("SERIES","").astype(str).str.upper().isin(["EQ","BE","BZ","BB"]))]  # EQ usually
    if sub.empty:
        return None
    val = pd.to_numeric(sub.iloc[0]["CLOSE"], errors="coerce")
    return None if pd.isna(val) else float(val)

# ---- BSE helpers ----
def _bse_bhav_url(d: date) -> str:
    # https://www.bseindia.com/download/BhavCopy/Equity/EQ100725_CSV.ZIP  (10 Jul 2025)
    return f"https://www.bseindia.com/download/BhavCopy/Equity/EQ{d.strftime('%d%m%y')}_CSV.ZIP"

def _load_bse_bhavcopy(d: date) -> Optional[pd.DataFrame]:
    url = _bse_bhav_url(d)
    r = _s_bse.get(url, timeout=25)
    if r.status_code == 404:
        return None
    r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
        names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        if not names:
            return None
        with zf.open(names[0]) as f:
            df = pd.read_csv(f)
    df.columns = [c.strip().upper().replace(" ", "_") for c in df.columns]
    return df

def _extract_close_bse(df: pd.DataFrame, bse_code: str) -> Optional[float]:
    if df is None:
        return None
    sc_col = "SC_CODE" if "SC_CODE" in df.columns else next((c for c in df.columns if "SC_CODE" in c), None)
    if sc_col is None or "CLOSE" not in df.columns:
        return None
    mask = df[sc_col].astype(str).str.strip() == str(bse_code)
    if not mask.any():
        return None
    val = pd.to_numeric(df.loc[mask, "CLOSE"].iloc[0], errors="coerce")
    return None if pd.isna(val) else float(val)

# ---- Unified lookups ----
def _get_close_any(d: date, nse_symbol: str, bse_code: Optional[str]) -> Optional[Tuple[str, float]]:
    """Try NSE first, then BSE. Return (EXCH, CLOSE) or None."""
    # NSE
    try:
        nse_df = _load_nse_bhavcopy(d)
        px = _extract_close_nse(nse_df, nse_symbol)
        if px is not None:
            return ("NSE", px)
    except Exception as e:
        print(f"  [warn] NSE {d}: {e}")
    # BSE
    if bse_code:
        try:
            bse_df = _load_bse_bhavcopy(d)
            px = _extract_close_bse(bse_df, bse_code)
            if px is not None:
                return ("BSE", px)
        except Exception as e:
            print(f"  [warn] BSE {d}: {e}")
    return None

def _find_T_on_or_before(result_d: date, nse_symbol: str, bse_code: Optional[str]) -> Tuple[date, str, float]:
    """Nearest trading day on/BEFORE result_d with a valid close; returns (date, exch, close)."""
    for back in range(SCAN_BACK_DAYS + 1):
        dt_try = result_d - timedelta(days=back)
        got = _get_close_any(dt_try, nse_symbol, bse_code)
        if got is not None:
            exch, px = got
            return dt_try, exch, px
        time.sleep(0.15)
    raise SystemExit(f"No bhavcopy found within {SCAN_BACK_DAYS} days before {result_d} for {nse_symbol}.")

def _find_next_trade_after(prev_d: date, nse_symbol: str, bse_code: Optional[str]) -> Optional[Tuple[date, str, float]]:
    for fwd in range(1, SCAN_FWD_DAYS + 1):
        dt_try = prev_d + timedelta(days=fwd)
        got = _get_close_any(dt_try, nse_symbol, bse_code)
        if got is not None:
            exch, px = got
            return dt_try, exch, px
        time.sleep(0.15)
    return None

def get_T_T1_T2_T3(symbol_nse: str, result_date_str: str) -> pd.DataFrame:
    """Implements your rules. result_date_str = DD-MM-YYYY."""
    bse_code = BSE_CODE_BY_NSE.get(symbol_nse)
    rdate = datetime.strptime(result_date_str, "%d-%m-%Y").date()
    print(f"[{symbol_nse}] Resolve around result date {rdate} (NSE first, then BSE)")
    # T
    t_date, t_exch, t_close = _find_T_on_or_before(rdate, symbol_nse, bse_code)
    print(f"  T   -> {t_date} ({t_exch}) close={t_close}")
    # T+1..T+3
    out = {
        "Ticker": symbol_nse,
        "Result Date": result_date_str,
        "_T_date": t_date.isoformat(),
        "Closing Price (T)": t_close,
        "_T1_date": None, "Closing Price (T+1)": None,
        "_T2_date": None, "Closing Price (T+2)": None,
        "_T3_date": None, "Closing Price (T+3)": None,
    }
    prev = t_date
    for k in (1, 2, 3):
        nxt = _find_next_trade_after(prev, symbol_nse, bse_code)
        if nxt is None:
            print(f"  T+{k} -> no trading day found within +{SCAN_FWD_DAYS}d window")
            break
        d_k, exch_k, px_k = nxt
        out[f"_T{k}_date"] = d_k.isoformat()
        out[f"Closing Price (T+{k})"] = px_k
        print(f"  T+{k} -> {d_k} ({exch_k}) close={px_k}")
        prev = d_k
    return pd.DataFrame([out])



In [21]:
# New Cell — Batch runner for TCS dates (show successes and failures)

import pandas as pd
from IPython.display import display

# Input dates (DD-MM-YYYY)
tcs_dates = [
    "08-07-2022",
    "10-10-2022",
    "09-01-2023",
    "12-04-2023",
    "12-07-2023",
    "11-10-2023",
    "11-01-2024",
    "12-04-2024",
    "11-07-2024",
    "10-10-2024",
    "09-01-2025",
    "10-04-2025",
    "10-07-2025",
]

success_rows = []
fail_rows = []

print(f"Running get_T_T1_T2_T3 for {len(tcs_dates)} TCS result dates...\n")

for i, dstr in enumerate(tcs_dates, start=1):
    print(f"[{i:02d}/{len(tcs_dates)}] TCS @ {dstr}")
    try:
        df_one = get_T_T1_T2_T3("TCS", dstr)  # uses the function from your previous cell
        success_rows.append(df_one)
    except SystemExit as e:
        # get_T_T1_T2_T3 raises SystemExit when no bhavcopy found in the scan window
        msg = str(e)
        print(f"  -> FAIL: {msg}")
        fail_rows.append({"Ticker": "TCS", "Result Date": dstr, "Reason": msg})
    except Exception as e:
        msg = f"{type(e).__name__}: {e}"
        print(f"  -> FAIL: {msg}")
        fail_rows.append({"Ticker": "TCS", "Result Date": dstr, "Reason": msg})

print("\n===== SUMMARY =====")
print(f"Success: {len(success_rows)}")
print(f"Failed : {len(fail_rows)}")

# Display successes
if success_rows:
    df_success = pd.concat(success_rows, ignore_index=True)
    # Nice ordering
    cols_order = [
        "Ticker", "Result Date",
        "_T_date", "Closing Price (T)",
        "_T1_date", "Closing Price (T+1)",
        "_T2_date", "Closing Price (T+2)",
        "_T3_date", "Closing Price (T+3)",
    ]
    # Keep any extra columns that function might add in future
    cols_order += [c for c in df_success.columns if c not in cols_order]
    df_success = df_success[cols_order]
    # Sort by the intended event date (Result Date parsed)
    df_success["_ResultDateParsed"] = pd.to_datetime(df_success["Result Date"], format="%d-%m-%Y", errors="coerce")
    df_success = df_success.sort_values("_ResultDateParsed").drop(columns=["_ResultDateParsed"])
    print("\n--- SUCCESSFUL LOOKUPS ---")
    display(df_success)
else:
    print("\n--- SUCCESSFUL LOOKUPS ---")
    print("(none)")

# Display failures
if fail_rows:
    df_fail = pd.DataFrame(fail_rows).sort_values("Result Date")
    print("\n--- FAILED LOOKUPS (no bhavcopy within window / other issue) ---")
    display(df_fail)
else:
    print("\n--- FAILED LOOKUPS ---")
    print("(none)")


Running get_T_T1_T2_T3 for 13 TCS result dates...

[01/13] TCS @ 08-07-2022
[TCS] Resolve around result date 2022-07-08 (NSE first, then BSE)
  T   -> 2022-07-08 (NSE) close=3265.45
  T+1 -> 2022-07-11 (NSE) close=3113.8
  T+2 -> 2022-07-12 (NSE) close=3084.7
  T+3 -> 2022-07-13 (NSE) close=3038.75
[02/13] TCS @ 10-10-2022
[TCS] Resolve around result date 2022-10-10 (NSE first, then BSE)
  T   -> 2022-10-10 (NSE) close=3118.55
  T+1 -> 2022-10-11 (NSE) close=3069.55
  T+2 -> 2022-10-12 (NSE) close=3100.75
  T+3 -> 2022-10-13 (NSE) close=3103.3
[03/13] TCS @ 09-01-2023
[TCS] Resolve around result date 2023-01-09 (NSE first, then BSE)
  T   -> 2023-01-09 (NSE) close=3319.95
  T+1 -> 2023-01-10 (NSE) close=3286.4
  T+2 -> 2023-01-11 (NSE) close=3328.7
  T+3 -> 2023-01-12 (NSE) close=3334.35
[04/13] TCS @ 12-04-2023
[TCS] Resolve around result date 2023-04-12 (NSE first, then BSE)
  T   -> 2023-04-12 (NSE) close=3241.65
  T+1 -> 2023-04-13 (NSE) close=3188.85
  T+2 -> 2023-04-17 (NSE) clos

C:\Users\cecme\AppData\Local\Temp\ipykernel_25736\1571504886.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_success = pd.concat(success_rows, ignore_index=True)


,Ticker,Result Date,_T_date,Closing Price (T),_T1_date,Closing Price (T+1),_T2_date,Closing Price (T+2),_T3_date,Closing Price (T+3)
0,TCS,08-07-2022,2022-07-08,"3,265.45",2022-07-11,"3,113.80",2022-07-12,"3,084.70",2022-07-13,"3,038.75"
1,TCS,10-10-2022,2022-10-10,"3,118.55",2022-10-11,"3,069.55",2022-10-12,"3,100.75",2022-10-13,"3,103.30"
2,TCS,09-01-2023,2023-01-09,"3,319.95",2023-01-10,"3,286.40",2023-01-11,"3,328.70",2023-01-12,"3,334.35"
3,TCS,12-04-2023,2023-04-12,"3,241.65",2023-04-13,"3,188.85",2023-04-17,"3,139.50",2023-04-18,"3,130.75"
4,TCS,12-07-2023,2023-07-12,"3,259.90",2023-07-13,"3,340.55",2023-07-14,"3,514.65",2023-07-17,"3,491.70"
5,TCS,11-10-2023,2023-10-11,"3,609.90",2023-10-12,"3,542.55",2023-10-13,"3,570.85",2023-10-16,"3,524.05"
6,TCS,11-01-2024,2024-01-11,"3,735.55",2024-01-12,"3,882.80",2024-01-15,"3,903.80",2024-01-16,"3,861.30"
7,TCS,12-04-2024,2024-04-12,"4,001.40",2024-04-15,"3,941.20",2024-04-16,"3,872.80",2024-04-18,"3,862.00"
8,TCS,11-07-2024,2024-07-05,"4,011.80",None,NaN,None,NaN,None,NaN
9,TCS,09-01-2025,2024-12-24,"4,180.65",None,NaN,None,NaN,None,NaN



--- FAILED LOOKUPS (no bhavcopy within window / other issue) ---


,Ticker,Result Date,Reason
1,TCS,10-04-2025,No bhavcopy found within 60 days before 2025-0...
2,TCS,10-07-2025,No bhavcopy found within 60 days before 2025-0...
0,TCS,10-10-2024,No bhavcopy found within 60 days before 2024-1...


In [26]:
# --------- CONFIG ---------
#INPUT_XLSX  = "Ticker_Quaterly_results_date.xlsx"   # attached file
INPUT_XLSX  = "Bajaj Auto Ticker_Quaterly_results_date.xlsx"   # attached file
OUTPUT_XLSX = "getQuaterlyPriceData_results_BajajAuto.xlsx"   # will be (over)written
DATE_FMT_IN = "%d-%m-%Y"  # what we pass into get_T_T1_T2_T3
# --------------------------

In [27]:
# New Cell — Batch price lookup from Excel, write results back to Excel

import os
import pandas as pd
import traceback
from datetime import datetime



# Sanity check: make sure function exists
try:
    _ = get_T_T1_T2_T3
except NameError as _e:
    raise SystemExit("get_T_T1_T2_T3() is not defined in this notebook. "
                     "Please run the cell that defines it first.") from _e

def _parse_to_ddmmyyyy(x):
    """
    Robust date parser:
    - accepts strings like '08-07-2022', '08/07/2022', '2022-07-08'
    - accepts Excel datetimes
    returns 'DD-MM-YYYY' string or None
    """
    if pd.isna(x):
        return None
    # If it's a pandas/Excel timestamp
    if isinstance(x, (pd.Timestamp, )):
        try:
            return x.strftime(DATE_FMT_IN)
        except Exception:
            pass
    # If it's a python datetime/date
    if hasattr(x, "strftime"):
        try:
            return x.strftime(DATE_FMT_IN)
        except Exception:
            pass
    # Try a few common string patterns
    s = str(x).strip()
    for dayfirst in (True, False):
        try:
            dt = pd.to_datetime(s, dayfirst=dayfirst, errors="raise")
            return dt.strftime(DATE_FMT_IN)
        except Exception:
            continue
    return None

print(f"Reading: {INPUT_XLSX}")
df_in = pd.read_excel(INPUT_XLSX)

# Normalize column names (strip spaces, title case on known names)
df_in.columns = [c.strip() for c in df_in.columns]
# Try common variants
col_map = {}
for c in df_in.columns:
    cl = c.lower()
    if cl == "ticker":
        col_map["Ticker"] = c
    elif cl in ("result date", "result_date", "date", "results date"):
        col_map["Result Date"] = c

missing_cols = [k for k in ("Ticker", "Result Date") if k not in col_map]
if missing_cols:
    raise SystemExit(f"Input file must have columns: {missing_cols} (case-insensitive). "
                     f"Found: {list(df_in.columns)}")

df_in = df_in.rename(columns={col_map["Ticker"]: "Ticker",
                              col_map["Result Date"]: "Result Date"})

# Clean rows
df_in["Ticker"] = df_in["Ticker"].astype(str).str.strip()
df_in["Result Date (DD-MM-YYYY)"] = df_in["Result Date"].apply(_parse_to_ddmmyyyy)
df_in = df_in[~df_in["Ticker"].eq("") & df_in["Result Date (DD-MM-YYYY)"].notna()].copy()

# Deduplicate (Ticker + date)
df_in = df_in.drop_duplicates(subset=["Ticker", "Result Date (DD-MM-YYYY)"]).reset_index(drop=True)

print(f"Total rows to process: {len(df_in)}")
if len(df_in) == 0:
    raise SystemExit("No valid (Ticker, Result Date) pairs after cleaning.")

success_parts = []
fail_rows = []

for i, row in df_in.iterrows():
    ticker = row["Ticker"]
    dstr   = row["Result Date (DD-MM-YYYY)"]
    print(f"[{i+1}/{len(df_in)}] {ticker} @ {dstr}")

    try:
        # Your function should return a small DataFrame with T, T+1, T+2, T+3 and dates
        df_one = get_T_T1_T2_T3(ticker, dstr)
        # Make sure we carry input columns for traceability
        if "Ticker" not in df_one.columns:
            df_one.insert(0, "Ticker", ticker)
        if "Result Date" not in df_one.columns:
            df_one.insert(1, "Result Date", dstr)
        success_parts.append(df_one)

    except SystemExit as e:
        msg = str(e)
        print(f"  -> NOT FOUND: {msg}")
        fail_rows.append({"Ticker": ticker, "Result Date": dstr, "Reason": msg})
    except Exception as e:
        msg = f"{type(e).__name__}: {e}"
        print(f"  -> ERROR: {msg}")
        # If you want the stacktrace in the console, uncomment next line:
        # traceback.print_exc()
        fail_rows.append({"Ticker": ticker, "Result Date": dstr, "Reason": msg})

# Combine successes
if success_parts:
    df_found = pd.concat(success_parts, ignore_index=True)
    # Sort by ticker then intended result date if present
    if "Result Date" in df_found.columns:
        try:
            _rdt = pd.to_datetime(df_found["Result Date"], dayfirst=True, errors="coerce")
            df_found = df_found.assign(_rdt=_rdt).sort_values(["Ticker", "_rdt"]).drop(columns=["_rdt"])
        except Exception:
            df_found = df_found.sort_values(["Ticker"])
else:
    df_found = pd.DataFrame()

# Combine fails
df_not_found = pd.DataFrame(fail_rows) if fail_rows else pd.DataFrame(columns=["Ticker", "Result Date", "Reason"])
if not df_not_found.empty:
    try:
        df_not_found = df_not_found.assign(_rdt=pd.to_datetime(df_not_found["Result Date"], dayfirst=True, errors="coerce")) \
                                   .sort_values(["Ticker", "_rdt"]) \
                                   .drop(columns=["_rdt"])
    except Exception:
        df_not_found = df_not_found.sort_values(["Ticker"])

# Write one Excel with two sheets
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    df_found.to_excel(writer, index=False, sheet_name="found")
    df_not_found.to_excel(writer, index=False, sheet_name="not_found")
print("\n==================== DONE ====================")
print(f"Found rows    : {len(df_found):>4}")
print(f"Not found rows: {len(df_not_found):>4}")
print(f"Wrote workbook: {OUTPUT_XLSX}")


Reading: Bajaj Auto Ticker_Quaterly_results_date.xlsx
Total rows to process: 13
[1/13] BAJAJ-AUTO @ 26-07-2022
[BAJAJ-AUTO] Resolve around result date 2022-07-26 (NSE first, then BSE)
  T   -> 2022-07-26 (NSE) close=3925.6
  T+1 -> 2022-07-27 (NSE) close=3883.85
  T+2 -> 2022-07-28 (NSE) close=3858.25
  T+3 -> 2022-07-29 (NSE) close=3914.45
[2/13] BAJAJ-AUTO @ 14-10-2022
[BAJAJ-AUTO] Resolve around result date 2022-10-14 (NSE first, then BSE)
  T   -> 2022-10-14 (NSE) close=3570.5
  T+1 -> 2022-10-17 (NSE) close=3629.0
  T+2 -> 2022-10-18 (NSE) close=3611.3
  T+3 -> 2022-10-19 (NSE) close=3655.85
[3/13] BAJAJ-AUTO @ 25-01-2023
[BAJAJ-AUTO] Resolve around result date 2023-01-25 (NSE first, then BSE)
  T   -> 2023-01-25 (NSE) close=3717.4
  T+1 -> 2023-01-27 (NSE) close=3936.75
  T+2 -> 2023-01-30 (NSE) close=3841.15
  T+3 -> 2023-01-31 (NSE) close=3818.25
[4/13] BAJAJ-AUTO @ 25-03-2023
[BAJAJ-AUTO] Resolve around result date 2023-03-25 (NSE first, then BSE)
  T   -> 2023-03-24 (NSE) clo

C:\Users\cecme\AppData\Local\Temp\ipykernel_25736\531769704.py:113: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_found = pd.concat(success_parts, ignore_index=True)
